In [ ]:
import numpy as np 
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt                               
import os 
import gc
import matplotlib.colors as mcolors
from matplotlib.ticker import MultipleLocator
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.gridspec as gridspec
matplotlib.rcParams["figure.dpi"] = 150
from particle import PDGID

In [ ]:
trackDir = '/home/mwells5/Muon_Collider_Smart_Pixels/Data_Files/Data_Set_2026Feb_copy_m/tl_with_moduleID_07_28_2026/'
flp = 0

In [ ]:
trackData = pd.DataFrame()
trackdata_list = []

trackHeader = ["cota", "cotb", "p", "flp", "ylocal", "zglobal", "pt", "t", "hit_pdg", "moduleID"]

countmm=0
countmp=0
for file in os.listdir(trackDir):
    if "bib_mm" in file:
        trackdata_list.append(pd.read_csv(f"{trackDir}{file}", sep=' ', names=trackHeader))
        countmm+=1 
    elif "bib_mp" in file: 
        trackdata_list.append(pd.read_csv(f"{trackDir}{file}", sep=' ', names=trackHeader))
        countmp+=1
    if countmp+countmm==200:
        break 

trackData = pd.concat(trackdata_list)
del trackdata_list
gc.collect()

trackData['adjusted_hit_time'] = trackData['t']-1e6*np.sqrt(trackData['zglobal']**2+30**2)/299792458

print(f"length of tracklist {len(trackData)}")
print(f"tracklist keys {trackData.keys()}")
print(trackData.head())

In [ ]:
def cut_tracks(tracks_df):
    tracks_df = tracks_df[(tracks_df['zglobal'] >= 0) & (tracks_df['zglobal'] <= 13)]
    tracks_df = tracks_df[(tracks_df['adjusted_hit_time'] >= -.09) & (tracks_df['adjusted_hit_time'] <= .15)]
    tracks_df = tracks_df[(tracks_df['moduleID'] == 1)]
    tracks_df.reset_index()

In [ ]:
def make_scatterplot(tracks_df):

    cut_tracks(tracks_df)
    
    x_l = (tracks_df['zglobal']%13)*40 # 1mm = 40px
    x_l = x_l.to_numpy()
    x_l = x_l.astype(int)
    
    y_l = (((tracks_df['ylocal'])+8.5)*40)-160 # changes range from 0 to 13 and convert to px
    y_l = y_l.to_numpy()
    y_l = y_l.astype(int)

    #momenta = tracks_df['p']
    #momenta = momenta.to_numpy()

    #print(momenta.min())
    #print(momenta.max())

    mod_array = np.ndarray((520,520))

    i = 0
    for i in range(tracks_df.shape[0]):
        if (x_l[i] in range(0, 520)) & (y_l[i] in range(0, 520)):
            mod_array[x_l[i], y_l[i]] += 1
        else:
            continue 

    fig, ax = plt.subplots(figsize=(7,7),dpi=200)
    im = ax.imshow(mod_array, interpolation='nearest', cmap='viridis',
                   vmin=0, vmax=1)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4%', pad=0.05)
    fig.colorbar(im, cax=cax, location='right',label='Momentum')
    ax.set_title("Scatterplot of a single module from tracks")
    
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4%', pad=0.05)
    fig.colorbar(im, cax=cax, location='right',label='Momentum')

    ax.set_xlim(0,520)
    ax.set_ylim(0,520)
    ax.set_xlabel("x-local [px]")
    ax.set_ylabel("y-local [px]")
    
    ax.xaxis.set_major_locator(ticker.MultipleLocator(40))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(40))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(20))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(20))
    plt.figtext(0.525,0.2, f"Occupied pixels: {percent_nonempty_pixels(mod_array)}%", fontsize=12, 
                    bbox=dict(facecolor='white', alpha=0.75))
    plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    plt.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)
        
    plt.tight_layout(pad=3.5)
    fig.canvas.draw()

make_scatterplot(trackData)